# 03 — Baseline Structural Analysis (Go/No-Go gate)

Loads `stats_base.json` and evaluates whether the base-model structural
statistics show enough prompt-level variation to justify the LoRA fine-tuning
experiment.

**Go/No-Go criterion (both must hold):**
1. At least **2** structural metrics have IQR > 0 and non-collapsed distributions.
2. A logistic regression classifier on structural metrics achieves **above-majority accuracy** for `correct vs incorrect` in cross-validation.

The final cell prints a clear GO / STOP recommendation.

**No GPU required.** Run locally.

## 0 — Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
else:
    print("WARNING: could not locate circuit_tracer repo.")

MY_WORK = _my_work if _root else Path(".").resolve()
STATS_FILE = MY_WORK / "results" / "statistics" / "stats_base.json"
print(f"Stats file: {STATS_FILE}")

## 1 — Load statistics

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from utils.graph_statistics import load_statistics, aggregate_statistics, _flatten_nested

all_stats = load_statistics(STATS_FILE)
stats = [s for s in all_stats if s.get("attribution_succeeded")]
stats_flat = [_flatten_nested(s) for s in stats]  # flattened for scalar access

print(f"Total entries  : {len(all_stats)}")
print(f"Succeeded      : {len(stats)}")
print(f"Success rate   : {len(stats)/len(all_stats):.1%}")
print()

agg = aggregate_statistics(all_stats)

# All scalar metrics (legacy + supervisor)
all_scalar_metrics = [
    # legacy
    'n_active_features', 'edge_density', 'mean_top50_score',
    'top10_over_top50', 'layer_entropy', 'mean_error_node_weight', 'logit_gap',
    # supervisor
    'layer_stats_mean', 'layer_stats_std', 'layer_stats_median', 'layer_stats_entropy_bits',
    'topk20_score_total', 'topk20_score_gini',
]

print(f"{'Metric':<36} {'Mean':>10} {'Std':>10} {'Median':>10} {'IQR':>10}")
print("-" * 80)
for m in all_scalar_metrics:
    v = agg.get(m)
    if v:
        print(f"{m:<36} {v['mean']:>10.4f} {v['std']:>10.4f} {v['median']:>10.4f} {v['iqr']:>10.4f}")
    else:
        print(f"{m:<36} {'N/A':>10}")

## 2 — Distribution plots for each metric

Check that distributions are non-collapsed (IQR > 0).

In [ ]:
def _is_correct(s):
    """Return True if model's logit prediction matches the ground-truth label."""
    if s.get("task_type", "binary") == "binary":
        pt = s.get("prob_true") or 0
        pf = s.get("prob_false") or 0
        return (pt > pf) == s["label"]
    else:
        return (s.get("prob_target") or 0) > 0.0  # any nonzero prob on correct token

plot_metrics = all_scalar_metrics  # 13 metrics total
ncols = 4
nrows = -(-len(plot_metrics) // ncols)  # ceiling division
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.flatten()

for ax, metric in zip(axes, plot_metrics):
    vals_flat = [(sf, s) for sf, s in zip(stats_flat, stats) if sf.get(metric) is not None]
    if not vals_flat:
        ax.set_title(f"{metric}\n(no data)")
        ax.axis('off')
        continue
    all_vals   = [sf[metric] for sf, _ in vals_flat]
    corr_vals  = [sf[metric] for sf, s in vals_flat if _is_correct(s)]
    wrong_vals = [sf[metric] for sf, s in vals_flat if not _is_correct(s)]
    arr = np.array(all_vals)
    q1, q3 = np.percentile(arr, [25, 75])

    ax.hist(arr, bins=20, alpha=0.6, color='steelblue', label='all')
    if corr_vals:
        ax.hist(corr_vals,  bins=20, alpha=0.55, color='green', label='correct')
    if wrong_vals:
        ax.hist(wrong_vals, bins=20, alpha=0.55, color='red',   label='incorrect')
    ax.axvline(q1, color='gray', linestyle='--', linewidth=0.8)
    ax.axvline(q3, color='gray', linestyle='--', linewidth=0.8)
    short = metric.replace('layer_stats_', 'ls_').replace('topk20_', 'k20_')
    ax.set_title(f"{short}\nIQR={q3-q1:.4f}", fontsize=8)
    ax.legend(fontsize=6)

# Hide any unused axes
for ax in axes[len(plot_metrics):]:
    ax.axis('off')

plt.tight_layout()
plt.suptitle("Base model: structural metric distributions (superset)", y=1.01, fontsize=12)

plot_path = MY_WORK / "results" / "statistics" / "base_metric_distributions.png"
plt.savefig(plot_path, bbox_inches="tight")
print(f"Saved: {plot_path}")
plt.show()

## 3 — Calibration: logit gap distribution by label

Shows whether the base model shows any separation between True and False prompts.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

gaps_true  = [s['logit_gap'] for s in stats if s.get('logit_gap') is not None and s['label']]
gaps_false = [s['logit_gap'] for s in stats if s.get('logit_gap') is not None and not s['label']]

if gaps_true:
    ax.hist(gaps_true,  bins=20, alpha=0.6, color='green', label='True prompts')
if gaps_false:
    ax.hist(gaps_false, bins=20, alpha=0.6, color='red',   label='False prompts')

ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
ax.set_xlabel("logit( True) − logit(False)")
ax.set_ylabel("Count")
ax.set_title("Calibration: logit gap by label (base model)")
ax.legend()
plt.tight_layout()
plt.savefig(MY_WORK / "results" / "statistics" / "base_logit_gap.png", bbox_inches="tight")
plt.show()

if gaps_true and gaps_false:
    from scipy import stats as scipy_stats
    stat_val, p_val = scipy_stats.mannwhitneyu(gaps_true, gaps_false, alternative='two-sided')
    print(f"Mann-Whitney U: U={stat_val:.1f}, p={p_val:.4f}")
    if p_val < 0.05:
        print("Base model shows statistically significant logit gap difference by label.")
    else:
        print("No significant logit gap difference by label (expected for base model).")

## 3b — Prune-curve and top-K=20 diagnostics

Visual checks for the supervisor-requested fingerprint metrics.

In [ ]:
# ── Prune-curve: mean node/edge counts per threshold across all prompts ────────
all_curves = [s.get("prune_curve") for s in stats if s.get("prune_curve")]
if all_curves:
    from collections import defaultdict
    bucket: dict = defaultdict(lambda: {"nodes": [], "edges": [], "density": []})
    for curve in all_curves:
        for pt in curve:
            t = pt["threshold"]
            if pt["n_nodes_kept"] is not None:
                bucket[t]["nodes"].append(pt["n_nodes_kept"])
            if pt["n_edges_kept"] is not None:
                bucket[t]["edges"].append(pt["n_edges_kept"])
            if pt["edge_density"] is not None:
                bucket[t]["density"].append(pt["edge_density"])

    thresholds = sorted(bucket.keys())
    mean_nodes   = [np.mean(bucket[t]["nodes"])   if bucket[t]["nodes"]   else np.nan for t in thresholds]
    mean_edges   = [np.mean(bucket[t]["edges"])   if bucket[t]["edges"]   else np.nan for t in thresholds]
    mean_density = [np.mean(bucket[t]["density"]) if bucket[t]["density"] else np.nan for t in thresholds]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))
    ax1.plot(thresholds, mean_nodes,   marker='o', color='steelblue')
    ax1.set_title("Prune curve: mean nodes kept"); ax1.set_xlabel("Threshold"); ax1.set_ylabel("Nodes")
    ax2.plot(thresholds, mean_edges,   marker='o', color='darkorange')
    ax2.set_title("Prune curve: mean edges kept"); ax2.set_xlabel("Threshold"); ax2.set_ylabel("Edges")
    ax3.plot(thresholds, mean_density, marker='o', color='seagreen')
    ax3.set_title("Prune curve: mean edge density"); ax3.set_xlabel("Threshold"); ax3.set_ylabel("Density")
    plt.tight_layout()
    plt.suptitle("Pruning survival curves (averaged over prompts)", y=1.02, fontsize=11)
    plt.savefig(MY_WORK / "results" / "statistics" / "base_prune_curves.png", bbox_inches="tight")
    plt.show()
else:
    print("No prune_curve data found in stats — was 02 run with the updated graph_statistics.py?")

# ── Top-K=20 layer histogram: mean distribution across prompts ─────────────────
from utils.graph_statistics import N_LAYERS
layer_hists = [s["topk20"]["layer_hist"] for s in stats if s.get("topk20")]
if layer_hists:
    mean_lh = np.mean(np.array(layer_hists), axis=0)  # shape (26,)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(N_LAYERS), mean_lh, color='mediumpurple', alpha=0.8)
    ax.set_xlabel("Layer"); ax.set_ylabel("Mean fraction of top-20 features")
    ax.set_title("Top-K=20 feature layer distribution (mean over prompts)")
    plt.tight_layout()
    plt.savefig(MY_WORK / "results" / "statistics" / "base_topk20_layer_hist.png", bbox_inches="tight")
    plt.show()

    # score_total distribution by label
    totals_true  = [s["topk20"]["score_total"] for s in stats if s.get("topk20") and s["label"]]
    totals_false = [s["topk20"]["score_total"] for s in stats if s.get("topk20") and not s["label"]]
    fig, ax = plt.subplots(figsize=(7, 4))
    if totals_true:
        ax.hist(totals_true,  bins=20, alpha=0.6, color='green', label='True prompts')
    if totals_false:
        ax.hist(totals_false, bins=20, alpha=0.6, color='red',   label='False prompts')
    ax.set_xlabel("topk20 score_total"); ax.set_ylabel("Count")
    ax.set_title("Top-K=20 influence concentration by label")
    ax.legend()
    plt.tight_layout()
    plt.savefig(MY_WORK / "results" / "statistics" / "base_topk20_score_by_label.png", bbox_inches="tight")
    plt.show()
else:
    print("No topk20 data found.")

## 4 — Logistic regression: correct vs incorrect from structural metrics (superset)

Condition 2 of the Go/No-Go gate: does a simple linear classifier using the
**full superset of scalar metrics** predict whether the model's first-token
prediction is correct?  Uses 5-fold stratified cross-validation.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Full superset of scalar features for the classifier
feature_cols = [
    # legacy
    'n_active_features', 'edge_density', 'mean_top50_score',
    'top10_over_top50', 'layer_entropy', 'mean_error_node_weight',
    # supervisor layer stats
    'layer_stats_mean', 'layer_stats_std', 'layer_stats_median', 'layer_stats_entropy_bits',
    # supervisor topk20
    'topk20_score_total', 'topk20_score_gini',
]

rows = []
for sf, s in zip(stats_flat, stats):
    row = [sf.get(c) for c in feature_cols]
    if any(v is None for v in row):
        continue
    correct = int(_is_correct(s))
    rows.append(row + [correct])

if not rows:
    print("No complete rows for classifier. Check stats_base.json is populated and 02 was run with the updated graph_statistics.py.")
    logreg_beats_majority = False
else:
    mat = np.array(rows)
    X = mat[:, :-1]
    y = mat[:, -1].astype(int)

    majority_acc = max(y.mean(), 1 - y.mean())

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=500, C=1.0)),
    ])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')

    print(f"Feature matrix shape : {X.shape}")
    print(f"Class balance        : {y.mean():.1%} correct")
    print(f"Majority baseline    : {majority_acc:.1%}")
    print(f"CV accuracy (5-fold) : {cv_scores.mean():.1%} ± {cv_scores.std():.1%}")
    print()

    pipe.fit(X, y)
    coefs = pipe.named_steps['clf'].coef_[0]
    print("Logistic regression coefficients (standardised features, sorted by |coef|):")
    for feat, coef in sorted(zip(feature_cols, coefs), key=lambda x: -abs(x[1])):
        print(f"  {feat:<36} {coef:>+.4f}")

    logreg_beats_majority = bool(cv_scores.mean() > majority_acc)
    print()
    print(f"Classifier beats majority: {logreg_beats_majority}")

## 5 — IQR spread check (Condition 1)

In [ ]:
# Check IQR spread for all superset scalar metrics (Condition 1 of Go/No-Go)
iqr_metrics = all_scalar_metrics  # defined in cell 4

non_trivial = []
print(f"{'Metric':<36} {'IQR':>12}  Status")
print("-" * 56)
for m in iqr_metrics:
    vals = [sf.get(m) for sf in stats_flat if sf.get(m) is not None]
    if vals:
        arr = np.array(vals, dtype=float)
        q1, q3 = np.percentile(arr, [25, 75])
        iqr = q3 - q1
        status = "OK" if iqr > 0 else "COLLAPSED"
        print(f"  {m:<34} {iqr:>12.6f}  [{status}]")
        if iqr > 0:
            non_trivial.append(m)
    else:
        print(f"  {m:<34} {'N/A':>12}  [NO DATA]")

print()
print(f"Metrics with IQR > 0: {len(non_trivial)}/{len(iqr_metrics)}")
condition1 = len(non_trivial) >= 2
print(f"Condition 1 (>=2 non-trivial metrics): {'PASS' if condition1 else 'FAIL'}")

## 6 — GO / STOP decision

In [ ]:
condition2 = logreg_beats_majority if 'logreg_beats_majority' in dir() else False

print("=" * 60)
print("GO / NO-GO DECISION")
print("=" * 60)
print(f"Condition 1 — >=2 metrics with IQR>0 : {'PASS' if condition1 else 'FAIL'}")
print(f"Condition 2 — classifier > majority  : {'PASS' if condition2 else 'FAIL'}")
print()

if condition1 and condition2:
    print("DECISION: GO — proceed to 04_lora_finetuning_triangle.ipynb")
elif condition1 and not condition2:
    print("DECISION: CONDITIONAL GO")
    print("  Structural variation exists but classifier signal is weak.")
    print("  Proceed with caution and note limitation in thesis.")
else:
    print("DECISION: STOP")
    print("  Structural statistics show insufficient variation across prompts.")
    print("  The structural-signal hypothesis is not supported on this setup.")
    print("  Do not proceed to LoRA fine-tuning.")